<a href="https://colab.research.google.com/github/ikhdaaakmalia/DAWN-YOLO/blob/main/DAWN-YOLO_TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.2 MB/s eta 0:00:00


In [ ]:
# YAML YOLOv8s + P2 head
yaml_content ="""
nc: 10
scales:
  s: [0.33, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]        # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]       # 1-P2/4
  - [-1, 3, C2f, [128, True]]        # 2
  - [-1, 1, Conv, [256, 3, 2]]       # 3-P3/8
  - [-1, 6, C2f, [256, True]]        # 4
  - [-1, 1, Conv, [512, 3, 2]]       # 5-P4/16
  - [-1, 6, C2f, [512, True]]        # 6
  - [-1, 1, Conv, [1024, 3, 2]]      # 7-P5/32
  - [-1, 3, C2f, [1024, True]]       # 8
  - [-1, 1, SPPF, [1024, 5]]         # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 10
  - [[-1, 6], 1, Concat, [1]]                     # 11 cat P4
  - [-1, 3, C2f, [512]]                           # 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 13
  - [[-1, 4], 1, Concat, [1]]                     # 14 cat P3
  - [-1, 3, C2f, [256]]                           # 15

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 16 ← BARU
  - [[-1, 2], 1, Concat, [1]]                     # 17 cat P2 ← BARU
  - [-1, 3, C2f, [128]]                           # 18 P2/4 ← BARU

  - [-1, 1, Conv, [128, 3, 2]]                    # 19
  - [[-1, 15], 1, Concat, [1]]                    # 20 cat P3
  - [-1, 3, C2f, [256]]                           # 21

  - [-1, 1, Conv, [256, 3, 2]]                    # 22
  - [[-1, 12], 1, Concat, [1]]                    # 23 cat P4
  - [-1, 3, C2f, [512]]                           # 24

  - [-1, 1, Conv, [512, 3, 2]]                    # 25
  - [[-1, 9], 1, Concat, [1]]                     # 26 cat P5
  - [-1, 3, C2f, [1024]]                          # 27

  - [[18, 21, 24, 27], 1, Detect, [nc]]           # 28 ← 4 head!
"""

YAML_PATH = '/content/drive/MyDrive/TA/models/yolov8s_p2.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)
print(f"YAML disimpan: {YAML_PATH}")

YAML disimpan: /content/drive/MyDrive/TA/models/yolov8s_p2.yaml


In [ ]:
import sys
mods = [k for k in sys.modules if 'ultralytics' in k]
for m in mods: del sys.modules[m]

from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/TA/models/yolov8s_p2.yaml')
params = sum(p.numel() for p in model.model.parameters())

print(f"✅ Model loaded!")
print(f"   Parameters : {params:,}")
print(f"   Expected   : ~11.5-12M")

# Cek jumlah head
from ultralytics.nn.modules.head import Detect
heads = [m for m in model.model.modules()
         if isinstance(m, Detect)]

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Model loaded!
   Parameters : 10,639,576
   Expected   : ~11.5-12M


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/TA/models/yolov8s_p2.yaml')

model.train(
    data         = '/content/drive/MyDrive/TA/visdrone/data_augmented_v2.yaml',
    epochs       = 100,
    imgsz        = 640,
    batch        = 8,
    optimizer    = 'Adam',
    lr0          = 0.01,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    patience     = 30,
    device       = 0,

    # Augmentasi
    hsv_v        = 0.5,    # brightness variation
    hsv_s        = 0.7,    # saturation variation
    hsv_h        = 0.015,  # hue variation
    erasing      = 0.5,    # random erasing (occlusion)
    fliplr       = 0.5,    # flip horizontal
    mosaic       = 1.0,    # mosaic
    scale        = 0.6,    # scale variation
    copy_paste   = 0.1,    # copy objek kecil

    project      = '/content/drive/MyDrive/TA/LAMYOLO/training',
    name         = 'exp9_p2_augv2',
    exist_ok     = True,
    resume       = False,
    verbose      = True,
)

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/TA/visdrone/data_augmented_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.5, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/TA/models/yolov8s_p2.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp9_p2_augv2, nbs=64, nms=False, opset=None, optimi

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/TA/LAMYOLO/training/'
             'exp9_p2_augv2/weights/last.pt')
model.train(resume=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/TA/visdrone/data_augmented_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.5, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, 

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/TA/LAMYOLO/training/'
             'exp9_p2_augv2/weights/last.pt')
model.train(resume=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/TA/visdrone/data_augmented_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.5, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bbd1d273590>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [ ]:
import os, cv2, yaml, shutil, json
import numpy as np
from pathlib import Path
from ultralytics import YOLO

MODEL_PATH   = "/content/drive/MyDrive/TA/LAMYOLO/training/exp9_p2_augv2/weights/best.pt"
DATA_YAML    = "/content/drive/MyDrive/TA/visdrone/data_augmented_v2.yaml"
TEST_IMG_DIR = "/content/drive/MyDrive/TA/visdrone/test/images"
TEST_LBL_DIR = "/content/drive/MyDrive/TA/visdrone/test/labels"
SAVE_DIR     = "/content/drive/MyDrive/TA/results"
os.makedirs(SAVE_DIR, exist_ok=True)

BRIGHT_THR = {"terang": 100, "remang": 40}

model = YOLO(MODEL_PATH)
print(f"✅ Model loaded: {MODEL_PATH}")

def get_brightness(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return 0
    return float(np.mean(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)))

def make_subset(img_list, cond_name):
    tmp_img = f"/tmp/eval9_{cond_name}/images"
    tmp_lbl = f"/tmp/eval9_{cond_name}/labels"
    if os.path.exists(f"/tmp/eval9_{cond_name}"):
        shutil.rmtree(f"/tmp/eval9_{cond_name}")
    os.makedirs(tmp_img)
    os.makedirs(tmp_lbl)
    for img_path in img_list:
        fname = os.path.basename(img_path)
        stem  = os.path.splitext(fname)[0]
        dst_img = f"{tmp_img}/{fname}"
        dst_lbl = f"{tmp_lbl}/{stem}.txt"
        if not os.path.exists(dst_img):
            os.symlink(img_path, dst_img)
        lbl_src = f"{TEST_LBL_DIR}/{stem}.txt"
        if os.path.exists(lbl_src) and not os.path.exists(dst_lbl):
            os.symlink(lbl_src, dst_lbl)
    with open(DATA_YAML) as f:
        cfg = yaml.safe_load(f)
    cfg["test"] = tmp_img
    tmp_yaml = f"/tmp/eval9_{cond_name}.yaml"
    with open(tmp_yaml, "w") as f:
        yaml.dump(cfg, f)
    return tmp_yaml

print("\n🔍 Mengklasifikasi citra test...")
all_imgs = list(Path(TEST_IMG_DIR).glob("*.jpg"))
splits   = {"overall": [], "terang": [], "remang": [], "gelap": []}

for img_path in all_imgs:
    mean = get_brightness(str(img_path))
    splits["overall"].append(str(img_path))
    if mean >= BRIGHT_THR["terang"]:
        splits["terang"].append(str(img_path))
    elif mean >= BRIGHT_THR["remang"]:
        splits["remang"].append(str(img_path))
    else:
        splits["gelap"].append(str(img_path))

for k, v in splits.items():
    print(f"  {k:8s}: {len(v):4d} gambar")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Model loaded: /content/drive/MyDrive/TA/LAMYOLO/training/exp9_p2_augv2/weights/best.pt

🔍 Mengklasifikasi citra test...
  overall : 1610 gambar
  terang  :  520 gambar
  remang  :  842 gambar
  gelap   :  248 gambar


In [ ]:
print(f"\n{'='*55}")
print("📊 EVALUASI")
print(f"{'='*55}")

results_exp9 = {}

for cond, img_list in splits.items():
    print(f"\n▶ Kondisi {cond.upper()} ({len(img_list)} gambar)...")

    if cond == "overall":
        m = model.val(
            data    = DATA_YAML,
            imgsz   = 640,
            conf    = 0.001,
            iou     = 0.5,
            device  = 0,
            verbose = False,
        )
    else:
        tmp_yaml = make_subset(img_list, cond)
        m = model.val(
            data    = tmp_yaml,
            imgsz   = 640,
            conf    = 0.001,
            iou     = 0.5,
            split   = "test",
            device  = 0,
            verbose = False,
        )

    p  = float(m.box.mp)
    r  = float(m.box.mr)
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0

    results_exp9[cond] = {
        "mAP50"    : round(float(m.box.map50) * 100, 2),
        "mAP50_95" : round(float(m.box.map)   * 100, 2),
        "precision": round(p  * 100, 2),
        "recall"   : round(r  * 100, 2),
        "f1"       : round(f1 * 100, 2),
    }

    print(f"  mAP@0.5   : {results_exp9[cond]['mAP50']}%")
    print(f"  mAP50-95  : {results_exp9[cond]['mAP50_95']}%")
    print(f"  Precision : {results_exp9[cond]['precision']}%")
    print(f"  Recall    : {results_exp9[cond]['recall']}%")
    print(f"  F1        : {results_exp9[cond]['f1']}%")

# Simpan
out_path = f"{SAVE_DIR}/exp9_results.json"
with open(out_path, "w") as f:
    json.dump(results_exp9, f, indent=2)
print(f"\n💾 Tersimpan: {out_path}")


📊 EVALUASI STANDAR EXP9

▶ Kondisi OVERALL (1610 gambar)...
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s_p2 summary (fused): 91 layers, 10,629,048 parameters, 0 gradients, 36.7 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 0.1±0.0 MB/s, size: 45.0 KB)
val: Scanning /content/drive/MyDrive/TA/visdrone/valid/labels.cache... 547 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 547/547 120.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 1.4s/it 49.1s
                   all        547      38717      0.471      0.387      0.392      0.228
Speed: 2.1ms preprocess, 13.6ms inference, 0.0ms loss, 5.2ms postprocess per image
Results saved to /content/runs/detect/val
  mAP@0.5   : 39.17%
  mAP50-95  : 22.81%
  Precision : 47.11%
  Recall    : 38.75%
  F1        : 42.52%

▶ Kondisi TERANG (520 gambar)...
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUD